# SQLite Intermedio
### Fundamentos de Gestión de Datos · TECSUP · Semana 3

**Docente:** Pilar Rocío Sayán Mejía · **Motor:** SQLite · **Entorno:** Google Colab (Python)

| | |
|---|---|
| **Nivel** | Intermedio (2 de 3: Básico → Intermedio → Avanzado) |
| **Requisito** | Haber terminado el cuaderno *SQLite Básico* |
| **Base de datos** | AynyTech Cusco completa: 4 tablas (`departamentos`, `empleados`, `proyectos`, `ventas`) |
| **Sentencias que aprenderás** | `INNER JOIN`, `LEFT JOIN`, `GROUP BY`, `HAVING`, `COUNT(DISTINCT ...)` |
| **Tiempo estimado** | 100 minutos |

## Cómo trabajar este cuaderno

1. Ejecuta las celdas **en orden**, de arriba hacia abajo.
2. No necesitas escribir código nuevo: **lee la consulta, ejecútala e interpreta** el resultado.
3. El cuaderno puede ejecutarse completo varias veces sin duplicar datos.
4. Al final hay **ejercicios** con solución escondida.

## 1. ¿Por qué relacionar las tablas?

Si guardas el **nombre del departamento** dentro de cada fila de `empleados` y ese nombre cambia, tendrías que corregir cientos de filas. Con tablas relacionadas, el nombre vive en **un solo lugar** (`departamentos`) y `empleados` solo guarda el número `id_depto`.

Eso ahorra espacio y evita contradicciones, pero tiene un costo: para ver el **nombre** del departamento junto al empleado, hay que **unir** las dos tablas. Eso hace `JOIN`.

### Las 4 tablas de AynyTech

| Tabla | ¿Qué guarda? | Filas |
|---|---|---|
| `departamentos` | Las áreas de la empresa | 6 |
| `empleados` | Los trabajadores y su departamento | 8 |
| `proyectos` | Proyectos asignados a un empleado responsable | 6 |
| `ventas` | Ventas registradas por cada empleado | 10 |

```
departamentos  ◄──  empleados  ◄──  proyectos
 (id_depto)          (id_depto)       (id_emp)      Cada proyecto lo lidera UN empleado.
                         ▲
                         └──────────  ventas
                                      (id_emp)      Cada venta la registra UN empleado.
```

La flecha indica que la tabla de la derecha tiene una **clave foránea** que apunta a la tabla de la izquierda.

## 2. Preparación del entorno y de la base de datos

Ejecuta la celda de preparación (carga `sqlite3` y `pandas`, y abre la conexión `conn`). Después se crean y llenan las tablas con `conn.executescript(...)`; las consultas se hacen con `pd.read_sql_query(..., conn)`. En SQLite se crea primero la tabla **padre** y después las **hijas**; `DROP TABLE IF EXISTS` (en orden inverso) permite repetir la celda sin errores.

In [1]:
# CELDA DE PREPARACIÓN: ejecútala una sola vez al inicio.
import sqlite3
import pandas as pd

conn = sqlite3.connect("aynitech_intermedio.db")      # crea (o abre) el archivo de la base de datos
conn.execute("PRAGMA foreign_keys = ON")   # SQLite necesita esta línea para respetar las claves foráneas

print("Versión de SQLite:", sqlite3.sqlite_version)

Versión de SQLite: 3.45.1


In [2]:
conn.executescript("""
DROP TABLE IF EXISTS ventas;
DROP TABLE IF EXISTS proyectos;
DROP TABLE IF EXISTS empleados;
DROP TABLE IF EXISTS departamentos;

CREATE TABLE departamentos (
    id_depto  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre    TEXT NOT NULL UNIQUE,
    ubicacion TEXT NOT NULL
);

CREATE TABLE empleados (
    id_emp    INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre    TEXT NOT NULL,
    cargo     TEXT NOT NULL,
    sueldo    REAL NOT NULL,
    email     TEXT UNIQUE,
    ciudad    TEXT DEFAULT 'Cusco',
    id_depto  INTEGER NOT NULL,
    FOREIGN KEY (id_depto) REFERENCES departamentos(id_depto)
);

CREATE TABLE proyectos (
    id_proyecto  INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre       TEXT NOT NULL,
    presupuesto  REAL NOT NULL,
    estado       TEXT DEFAULT 'En curso',     -- En curso, Terminado o Pausado
    id_emp       INTEGER NOT NULL,            -- empleado responsable
    FOREIGN KEY (id_emp) REFERENCES empleados(id_emp)
);

CREATE TABLE ventas (
    id_venta  INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha     TEXT NOT NULL,                  -- formato 'AAAA-MM-DD'
    monto     REAL NOT NULL,                  -- monto en soles
    producto  TEXT NOT NULL,
    id_emp    INTEGER NOT NULL,               -- empleado que registró la venta
    FOREIGN KEY (id_emp) REFERENCES empleados(id_emp)
);
""")
print("Tablas creadas correctamente")

Tablas creadas correctamente


In [3]:
conn.executescript("""
INSERT INTO departamentos (nombre, ubicacion) VALUES
    ('Sistemas',       'Piso 3 - Av. El Sol'),   -- id_depto = 1
    ('Ventas',         'Piso 1 - Av. El Sol'),   -- id_depto = 2
    ('Administracion', 'Piso 2 - Av. El Sol'),   -- id_depto = 3
    ('Soporte',        'Piso 1 - Urb. Ttio'),    -- id_depto = 4
    ('Proyectos',      'Piso 4 - Av. El Sol'),   -- id_depto = 5
    ('Marketing',      'Piso 2 - Urb. Ttio');    -- id_depto = 6  (todavía sin empleados)

INSERT INTO empleados (nombre, cargo, sueldo, email, ciudad, id_depto) VALUES
    ('Carlos Quispe', 'Desarrollador',   3500.00, 'cquispe@ayni.pe',  'Cusco', 1),   -- id_emp = 1
    ('Lucia Mamani',  'Analista',        2800.00, 'lmamani@ayni.pe',  'Cusco', 1),   -- id_emp = 2
    ('Pedro Condori', 'Jefe de Ventas',  4200.00, 'pcondori@ayni.pe', 'Cusco', 2),   -- id_emp = 3
    ('Ana Huanca',    'Vendedora',       2200.00, 'ahuanca@ayni.pe',  'Puno',  2),   -- id_emp = 4
    ('Jose Ttito',    'Administrador',   3100.00, NULL,               'Cusco', 3),   -- id_emp = 5
    ('Maria Ccopa',   'Soporte TI',      2600.00, 'mccopa@ayni.pe',   'Cusco', 4),   -- id_emp = 6
    ('Roberto Apaza', 'Desarrollador',   3800.00, 'rapaza@ayni.pe',   'Cusco', 1),   -- id_emp = 7
    ('Sofia Huallpa', 'Lider Proyectos', 4500.00, 'shuallpa@ayni.pe', 'Cusco', 5);   -- id_emp = 8

INSERT INTO proyectos (nombre, presupuesto, estado, id_emp) VALUES
    ('Sistema ERP Cusco',     85000.00, 'En curso',   1),   -- Carlos Quispe
    ('App Movil Turismo',     42000.00, 'Terminado',  7),   -- Roberto Apaza
    ('Portal Web Ventas',     30000.00, 'En curso',   8),   -- Sofia Huallpa
    ('Migracion Base Datos',  55000.00, 'Pausado',    2),   -- Lucia Mamani
    ('Chatbot Soporte',       18000.00, 'En curso',   6),   -- Maria Ccopa
    ('Dashboard Gerencial',   25000.00, 'Terminado',  8);   -- Sofia Huallpa

INSERT INTO ventas (fecha, monto, producto, id_emp) VALUES
    ('2024-01-15', 12500.00, 'Licencia Software',     3),   -- Pedro Condori
    ('2024-02-03',  8200.00, 'Consultoria TI',        3),   -- Pedro Condori
    ('2024-02-20', 15000.00, 'Desarrollo Web',        1),   -- Carlos Quispe
    ('2024-03-10',  5500.00, 'Soporte Anual',         6),   -- Maria Ccopa
    ('2024-03-25', 22000.00, 'ERP Modulo RRHH',       1),   -- Carlos Quispe
    ('2024-04-05',  9800.00, 'App Movil',             4),   -- Ana Huanca
    ('2024-04-18', 31000.00, 'Sistema Contable',      3),   -- Pedro Condori
    ('2024-05-02',  4500.00, 'Capacitacion SQLite',   2),   -- Lucia Mamani
    ('2024-05-15', 17500.00, 'Plataforma eLearning',  7),   -- Roberto Apaza
    ('2024-06-01', 11200.00, 'Dashboard BI',          8);   -- Sofia Huallpa
""")
print("Datos insertados correctamente")

Datos insertados correctamente


### Comprobar las 4 tablas

Esta consulta junta cuatro resultados en uno solo con `UNION ALL` (lo verás con detalle en el manual avanzado). Debe mostrar 6, 8, 6 y 10 filas.

In [4]:
pd.read_sql_query("""
SELECT 'departamentos' AS Tabla, COUNT(*) AS Filas FROM departamentos
UNION ALL SELECT 'empleados',    COUNT(*) FROM empleados
UNION ALL SELECT 'proyectos',    COUNT(*) FROM proyectos
UNION ALL SELECT 'ventas',       COUNT(*) FROM ventas
""", conn)

,Tabla,Filas
0,departamentos,6
1,empleados,8
2,proyectos,6
3,ventas,10


## 3. `INNER JOIN`: unir tablas

La tabla `empleados` guarda `id_depto` (un número) pero **no el nombre** del departamento. `INNER JOIN` combina las filas de dos tablas **cuando un valor coincide** (normalmente clave foránea = clave primaria).

```sql
SELECT t1.columna, t2.columna   -- de qué tabla viene cada columna
FROM   tabla1 t1                -- tabla1 con alias t1 (nombre corto)
INNER JOIN tabla2 t2            -- solo filas que coinciden en AMBAS tablas
       ON t1.id = t2.id_fk      -- ON: la condición de unión
WHERE  condicion;               -- filtros opcionales
```

> **Alias:** `empleados e` significa «llamaré `e` a la tabla `empleados` en esta consulta». Así escribes `e.nombre` en lugar de `empleados.nombre`.

### 3.1 Empleados con el nombre de su departamento

In [5]:
pd.read_sql_query("""
SELECT e.nombre       AS Empleado,
       e.cargo        AS Cargo,
       e.sueldo       AS Sueldo,
       d.nombre       AS Departamento,     -- d = alias de departamentos
       d.ubicacion    AS Ubicacion
FROM   empleados e                         -- tabla principal
INNER JOIN departamentos d                 -- la unimos con departamentos...
       ON e.id_depto = d.id_depto          -- ...donde los id_depto coinciden
ORDER BY d.nombre, e.sueldo DESC           -- por departamento y luego por sueldo
""", conn)

,Empleado,Cargo,Sueldo,Departamento,Ubicacion
0,Jose Ttito,Administrador,3100.0,Administracion,Piso 2 - Av. El Sol
1,Sofia Huallpa,Lider Proyectos,4500.0,Proyectos,Piso 4 - Av. El Sol
2,Roberto Apaza,Desarrollador,3800.0,Sistemas,Piso 3 - Av. El Sol
3,Carlos Quispe,Desarrollador,3500.0,Sistemas,Piso 3 - Av. El Sol
4,Lucia Mamani,Analista,2800.0,Sistemas,Piso 3 - Av. El Sol
5,Maria Ccopa,Soporte TI,2600.0,Soporte,Piso 1 - Urb. Ttio
6,Pedro Condori,Jefe de Ventas,4200.0,Ventas,Piso 1 - Av. El Sol
7,Ana Huanca,Vendedora,2200.0,Ventas,Piso 1 - Av. El Sol


### 3.2 Reporte de ventas con el nombre del vendedor

In [6]:
pd.read_sql_query("""
SELECT v.fecha     AS Fecha,
       e.nombre    AS Vendedor,
       v.producto  AS Producto,
       v.monto     AS Monto_Soles
FROM   ventas v
INNER JOIN empleados e ON v.id_emp = e.id_emp
ORDER BY v.monto DESC                      -- de mayor a menor monto
""", conn)

,Fecha,Vendedor,Producto,Monto_Soles
0,2024-04-18,Pedro Condori,Sistema Contable,31000.0
1,2024-03-25,Carlos Quispe,ERP Modulo RRHH,22000.0
2,2024-05-15,Roberto Apaza,Plataforma eLearning,17500.0
3,2024-02-20,Carlos Quispe,Desarrollo Web,15000.0
4,2024-01-15,Pedro Condori,Licencia Software,12500.0
5,2024-06-01,Sofia Huallpa,Dashboard BI,11200.0
6,2024-04-05,Ana Huanca,App Movil,9800.0
7,2024-02-03,Pedro Condori,Consultoria TI,8200.0
8,2024-03-10,Maria Ccopa,Soporte Anual,5500.0
9,2024-05-02,Lucia Mamani,Capacitacion SQLite,4500.0


### 3.3 `JOIN` de tres tablas

Se pueden encadenar varios `JOIN`. Aquí unimos **proyectos → empleados → departamentos** para saber, de cada proyecto activo, quién lo lidera y en qué área trabaja.

In [7]:
pd.read_sql_query("""
SELECT e.nombre       AS Responsable,
       d.nombre       AS Departamento,
       p.nombre       AS Proyecto,
       p.presupuesto  AS Presupuesto,
       p.estado       AS Estado
FROM   proyectos p
INNER JOIN empleados     e ON p.id_emp   = e.id_emp
INNER JOIN departamentos d ON e.id_depto = d.id_depto
WHERE  p.estado = 'En curso'               -- solo proyectos activos
ORDER BY p.presupuesto DESC
""", conn)

,Responsable,Departamento,Proyecto,Presupuesto,Estado
0,Carlos Quispe,Sistemas,Sistema ERP Cusco,85000.0,En curso
1,Sofia Huallpa,Proyectos,Portal Web Ventas,30000.0,En curso
2,Maria Ccopa,Soporte,Chatbot Soporte,18000.0,En curso


## 4. `LEFT JOIN`: conservar todo lo de la izquierda

`INNER JOIN` **descarta** las filas sin pareja. `LEFT JOIN` conserva **todas las filas de la tabla izquierda** (la del `FROM`) y, cuando no hay pareja en la derecha, rellena las columnas de la derecha con `NULL`.

| Tipo de unión | ¿Qué devuelve? | Filas sin pareja |
|---|---|---|
| `INNER JOIN` | Solo filas con coincidencia en **ambas** tablas | Se descartan |
| `LEFT JOIN` | **Todas** las de la izquierda + coincidencias de la derecha | `NULL` en las columnas de la derecha |

### 4.1 Departamentos con y sin empleados

El departamento **Marketing** todavía no tiene empleados. Compara los dos resultados.

In [8]:
print("INNER JOIN: Marketing NO aparece (no tiene pareja):")
pd.read_sql_query("""
SELECT d.nombre AS Departamento, e.nombre AS Empleado
FROM   departamentos d
INNER JOIN empleados e ON d.id_depto = e.id_depto
ORDER BY d.nombre, e.nombre
""", conn)

INNER JOIN: Marketing NO aparece (no tiene pareja):


,Departamento,Empleado
0,Administracion,Jose Ttito
1,Proyectos,Sofia Huallpa
2,Sistemas,Carlos Quispe
3,Sistemas,Lucia Mamani
4,Sistemas,Roberto Apaza
5,Soporte,Maria Ccopa
6,Ventas,Ana Huanca
7,Ventas,Pedro Condori


In [9]:
print("LEFT JOIN: Marketing SÍ aparece, con Empleado vacío (NULL):")
pd.read_sql_query("""
SELECT d.nombre AS Departamento, e.nombre AS Empleado
FROM   departamentos d
LEFT JOIN empleados e ON d.id_depto = e.id_depto
ORDER BY d.nombre, e.nombre
""", conn)

LEFT JOIN: Marketing SÍ aparece, con Empleado vacío (NULL):


,Departamento,Empleado
0,Administracion,Jose Ttito
1,Marketing,NaN
2,Proyectos,Sofia Huallpa
3,Sistemas,Carlos Quispe
4,Sistemas,Lucia Mamani
5,Sistemas,Roberto Apaza
6,Soporte,Maria Ccopa
7,Ventas,Ana Huanca
8,Ventas,Pedro Condori


### 4.2 Encontrar lo que no tiene pareja: `LEFT JOIN` + `IS NULL`

Truco muy usado: después de un `LEFT JOIN`, filtrar `WHERE columna_de_la_derecha IS NULL` deja **solo las filas que no tienen pareja**. Preguntas típicas: *¿qué departamentos no tienen empleados? ¿qué empleados no han vendido nada?*

In [10]:
print("Departamentos sin empleados:")
pd.read_sql_query("""
SELECT d.nombre AS Departamento_Vacio
FROM   departamentos d
LEFT JOIN empleados e ON d.id_depto = e.id_depto
WHERE  e.id_emp IS NULL
""", conn)

Departamentos sin empleados:


,Departamento_Vacio
0,Marketing


In [11]:
print("Empleados que no registraron ninguna venta:")
pd.read_sql_query("""
SELECT e.nombre AS Empleado_Sin_Ventas, e.cargo AS Cargo
FROM   empleados e
LEFT JOIN ventas v ON e.id_emp = v.id_emp
WHERE  v.id_venta IS NULL
""", conn)

Empleados que no registraron ninguna venta:


,Empleado_Sin_Ventas,Cargo
0,Jose Ttito,Administrador


## 5. `GROUP BY`: agrupar y resumir

Las funciones de resumen (`COUNT`, `SUM`, `AVG`, `MAX`, `MIN`) del nivel básico calculaban **un solo valor para toda la tabla**. Con `GROUP BY` se calcula **un valor por cada grupo**.

> **Regla de oro:** toda columna del `SELECT` que **no** sea una función de resumen debe aparecer en el `GROUP BY`.

### 5.1 Empleados y planilla por departamento

Con `LEFT JOIN` desde `departamentos`, **Marketing aparece con 0 empleados**. Observa que se cuenta `COUNT(e.id_emp)` (no `COUNT(*)`): `COUNT(*)` contaría la fila vacía de Marketing como si fuera un empleado.

In [12]:
pd.read_sql_query("""
SELECT d.nombre                    AS Departamento,
       COUNT(e.id_emp)             AS Num_Empleados,
       ROUND(AVG(e.sueldo), 2)     AS Sueldo_Promedio,
       SUM(e.sueldo)               AS Total_Planilla
FROM   departamentos d
LEFT JOIN empleados e ON d.id_depto = e.id_depto
GROUP BY d.nombre                  -- un grupo por departamento
ORDER BY Total_Planilla DESC
""", conn)

,Departamento,Num_Empleados,Sueldo_Promedio,Total_Planilla
0,Sistemas,3,3366.67,10100.0
1,Ventas,2,3200.00,6400.0
2,Proyectos,1,4500.00,4500.0
3,Administracion,1,3100.00,3100.0
4,Soporte,1,2600.00,2600.0
5,Marketing,0,NaN,NaN


### 5.2 Ventas por vendedor

In [13]:
pd.read_sql_query("""
SELECT e.nombre         AS Vendedor,
       COUNT(v.id_venta) AS Num_Ventas,
       SUM(v.monto)      AS Total_Vendido
FROM   ventas v
INNER JOIN empleados e ON v.id_emp = e.id_emp
GROUP BY e.nombre
ORDER BY Total_Vendido DESC
""", conn)

,Vendedor,Num_Ventas,Total_Vendido
0,Pedro Condori,3,51700.0
1,Carlos Quispe,2,37000.0
2,Roberto Apaza,1,17500.0
3,Sofia Huallpa,1,11200.0
4,Ana Huanca,1,9800.0
5,Maria Ccopa,1,5500.0
6,Lucia Mamani,1,4500.0


### 5.3 Proyectos por estado

In [14]:
pd.read_sql_query("""
SELECT estado                 AS Estado,
       COUNT(*)               AS Num_Proyectos,
       SUM(presupuesto)       AS Presupuesto_Total
FROM   proyectos
GROUP BY estado
ORDER BY Presupuesto_Total DESC
""", conn)

,Estado,Num_Proyectos,Presupuesto_Total
0,En curso,3,133000.0
1,Terminado,2,67000.0
2,Pausado,1,55000.0


### 5.4 `COUNT(DISTINCT ...)`: contar sin repetir

`COUNT(DISTINCT columna)` cuenta **cuántos valores diferentes** hay. Aquí: ¿cuántos vendedores distintos hubo en total y cuántas ventas?

In [15]:
pd.read_sql_query("""
SELECT COUNT(*)              AS Ventas,
       COUNT(DISTINCT id_emp) AS Vendedores_Distintos
FROM   ventas
""", conn)

,Ventas,Vendedores_Distintos
0,10,7


## 6. `HAVING`: filtrar grupos (y su diferencia con `WHERE`)

| | `WHERE` | `HAVING` |
|---|---|---|
| **¿Cuándo filtra?** | **Antes** de agrupar | **Después** de agrupar |
| **¿Sobre qué actúa?** | Filas individuales | Resultados de grupos (`SUM`, `COUNT`...) |
| **¿Puede usar `SUM`, `COUNT`...?** | No | Sí |

> **Regla práctica:** si el filtro usa una función de resumen (`SUM`, `COUNT`...), usa `HAVING`.

### 6.1 `HAVING`: vendedores que vendieron más de S/ 20 000

In [16]:
pd.read_sql_query("""
SELECT e.nombre         AS Vendedor,
       COUNT(v.id_venta) AS Num_Ventas,
       SUM(v.monto)      AS Total_Vendido
FROM   ventas v
INNER JOIN empleados e ON v.id_emp = e.id_emp
GROUP BY e.nombre
HAVING SUM(v.monto) > 20000      -- filtra los GRUPOS ya calculados
ORDER BY Total_Vendido DESC
""", conn)

,Vendedor,Num_Ventas,Total_Vendido
0,Pedro Condori,3,51700.0
1,Carlos Quispe,2,37000.0


### 6.2 `WHERE` y `HAVING` en la misma consulta

Aquí `WHERE` descarta primero las ventas pequeñas (menores a S/ 10 000) y **después** se agrupa. Compara con el resultado anterior: cambian los totales porque se sumaron menos ventas.

In [17]:
pd.read_sql_query("""
SELECT e.nombre          AS Vendedor,
       COUNT(v.id_venta) AS Num_Ventas,
       SUM(v.monto)      AS Total_Vendido
FROM   ventas v
INNER JOIN empleados e ON v.id_emp = e.id_emp
WHERE  v.monto >= 10000          -- 1.º: se filtran FILAS
GROUP BY e.nombre                -- 2.º: se agrupa
HAVING SUM(v.monto) > 20000      -- 3.º: se filtran GRUPOS
ORDER BY Total_Vendido DESC
""", conn)

,Vendedor,Num_Ventas,Total_Vendido
0,Pedro Condori,2,43500.0
1,Carlos Quispe,2,37000.0


## 7. Orden correcto de las cláusulas

```
SELECT   (qué columnas mostrar)
FROM     (de qué tabla o tablas)
JOIN     (unir con otras tablas)
WHERE    (filtrar filas  — ANTES de agrupar)
GROUP BY (agrupar resultados)
HAVING   (filtrar grupos — DESPUÉS de agrupar)
ORDER BY (ordenar el resultado final)
LIMIT    (limitar la cantidad de filas)
```

### Errores frecuentes

| Error | Cómo se ve | Solución |
|---|---|---|
| Columna sin agrupar | Resultado engañoso (SQLite elige un valor al azar) | Poner la columna en el `GROUP BY` |
| `SUM(...)` dentro de `WHERE` | `misuse of aggregate` | Usar `HAVING` |
| `INNER JOIN` cuando necesitas todo | Faltan filas (por ejemplo, Marketing) | Usar `LEFT JOIN` |
| `COUNT(*)` después de un `LEFT JOIN` | Cuenta la fila vacía como 1 | Usar `COUNT(columna_de_la_derecha)` |
| Olvidar el `ON` | Cada fila se une con todas las demás | Escribir siempre la condición `ON` |

## 8. Resumen del nivel intermedio

| Comando | ¿Para qué sirve? | Ejemplo |
|---|---|---|
| `INNER JOIN ... ON` | Une tablas; solo filas con pareja | `INNER JOIN departamentos d ON e.id_depto = d.id_depto` |
| `LEFT JOIN ... ON` | Une tablas; conserva todo lo de la izquierda | `LEFT JOIN ventas v ON e.id_emp = v.id_emp` |
| `LEFT JOIN` + `IS NULL` | Encuentra filas sin pareja | `WHERE v.id_venta IS NULL` |
| `GROUP BY` | Un resultado por grupo | `GROUP BY d.nombre` |
| `HAVING` | Filtra grupos | `HAVING SUM(monto) > 20000` |
| `COUNT(DISTINCT x)` | Cuenta valores distintos | `COUNT(DISTINCT id_emp)` |
| Alias (`AS`, `e`, `d`) | Nombres cortos y claros | `empleados e`, `d.nombre AS Departamento` |

En el **nivel avanzado** verás subconsultas, vistas, `CASE`, `UNION`, `RIGHT JOIN` y `FULL JOIN`.

## 9. Ejercicios

Escribe tu consulta en una celda nueva con `pd.read_sql_query("""...""", conn)`. Cada ejercicio indica cuántas filas debe devolver. La solución está escondida.

---

**Ejercicio 1.** Muestra la fecha, el vendedor, el producto y el monto de las ventas de los empleados del departamento **Sistemas** (une `ventas`, `empleados` y `departamentos`). *Debe devolver 4 filas.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT v.fecha, e.nombre AS Vendedor, v.producto, v.monto
FROM   ventas v
INNER JOIN empleados     e ON v.id_emp   = e.id_emp
INNER JOIN departamentos d ON e.id_depto = d.id_depto
WHERE  d.nombre = 'Sistemas'
ORDER BY v.fecha
""", conn)
```
</details>

---

**Ejercicio 2.** ¿Qué departamentos **no tienen empleados**? *Debe devolver 1 fila: Marketing.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT d.nombre AS Departamento
FROM   departamentos d
LEFT JOIN empleados e ON d.id_depto = e.id_depto
WHERE  e.id_emp IS NULL
""", conn)
```
</details>

---

**Ejercicio 3.** Calcula el **total vendido por departamento**, de mayor a menor. *Debe devolver 4 filas; el primero es Ventas con 61500.0.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT d.nombre AS Departamento, SUM(v.monto) AS Total_Vendido
FROM   ventas v
INNER JOIN empleados     e ON v.id_emp   = e.id_emp
INNER JOIN departamentos d ON e.id_depto = d.id_depto
GROUP BY d.nombre
ORDER BY Total_Vendido DESC
""", conn)
```
</details>

---

**Ejercicio 4.** Muestra cuántos proyectos hay por **estado** y su presupuesto total. *Debe devolver 3 filas; «En curso» suma 133000.0.*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT estado, COUNT(*) AS Num_Proyectos, SUM(presupuesto) AS Presupuesto_Total
FROM   proyectos
GROUP BY estado
""", conn)
```
</details>

---

**Ejercicio 5.** Muestra solo los vendedores con **dos o más ventas**. *Debe devolver 2 filas: Pedro Condori (3) y Carlos Quispe (2).*

<details><summary>Ver solución</summary>

```python
pd.read_sql_query("""
SELECT e.nombre AS Vendedor, COUNT(v.id_venta) AS Num_Ventas
FROM   ventas v
INNER JOIN empleados e ON v.id_emp = e.id_emp
GROUP BY e.nombre
HAVING COUNT(v.id_venta) >= 2
ORDER BY Num_Ventas DESC
""", conn)
```
</details>

---

*SQLite Intermedio · TECSUP · Fundamentos de Gestión de Datos · Docente: Pilar Rocío Sayán Mejía*